# Let's start looking into building a compound generator.

Recall that this project requires 3 components:
1. Grader - chemprop prediction model
2. Generator - ChemTSV2 novel compound generator
3. Rewarder - custom class that interprates the Grader's output and feeds back to the Generator.

ChemTSV2 article: https://wires.onlinelibrary.wiley.com/doi/10.1002/wcms.1680

Repo: https://github.com/molecule-generator-collection/ChemTSv2

Docs: https://molecule-generator-collection.github.io/ChemTSv2/

### Citations:

Ishida, S. and Aasawat, T. and Sumita, M. and Katouda, M. and Yoshizawa, T. and Yoshizoe, K. and Tsuda, K. and Terayama, K. (2023). ChemTSv2: Functional molecular design using de novo molecule generator. WIREs Computational Molecular Science https://wires.onlinelibrary.wiley.com/doi/10.1002/wcms.1680 ↩

Yang, X., Zhang, J., Yoshizoe, K., Terayama, K., & Tsuda, K. (2017). ChemTS: an efficient python library for de novo molecular generation. Science and Technology of Advanced Materials, 18(1), 972–976. https://doi.org/10.1080/14686996.2017.1401424 ↩

Yang, X., Aasawat, T., & Yoshizoe, K. (2021). Practical Massively Parallel Monte-Carlo Tree Search Applied to Molecular Design. In International Conference on Learning Representations. https://openreview.net/forum?id=6k7VdojAIK ↩

**Switch environment to Python 11 with GPU (cuda). You will most likely have to run this notebook a second time due to numpy updates**

In [ ]:
!python --version

Python 3.11.13


In [ ]:
!pip install tensorflow==2.14.1

In [ ]:
!pip install chemtsv2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 3.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.1.4 which is incompatible.
mizani 0.13.5 requires pandas>=2.2.0, but you have pandas 2.1.4 which is incompatible.
plotnine 0.14.6 requires pandas>=2.2.0, but you have pandas 2.1.4 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.14.1 which is incompatible.


In [ ]:
!pip install chemprop

  Using cached chemprop-2.2.2-py3-none-any.whl.metadata (9.6 kB)
  Using cached lightning-2.6.1-py3-none-any.whl.metadata (44 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.4/150.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.8 MB/s et

### Let's first create a reward system and turn it into a .py file

In [ ]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_reward.py
import sys
import numpy as np
from rdkit.Chem import Descriptors
from chemtsv2.abc import Reward
from chemprop.models.model import MPNN
from chemprop import data, featurizers, models
from lightning import pytorch as pl
from rdkit import Chem
import torch


def predict_chemprop(mol):
  # turn mol to smiles
  smiles = []
  smile = Chem.MolToSmiles(mol)
  smiles.append(smile)

  # load model
  checkpoint_path = '/content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/models/best-epoch=52-val_loss=0.33.ckpt' # CHANGE THIS TO WHERE THE MODEL .CKPT FILE IS SAVED
  mpnn = MPNN.load_from_checkpoint(checkpoint_path)

  # assign data
  test_data = [data.MoleculeDatapoint.from_smi(smi) for smi in smiles]

  # set model vars
  featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
  test_dset = data.MoleculeDataset(test_data, featurizer=featurizer)
  test_loader = data.build_dataloader(test_dset, shuffle=False)

  # predict
  with torch.inference_mode():
      trainer = pl.Trainer(
          logger=None,
          enable_progress_bar=True,
          accelerator="cuda",
          devices=1
      )
      test_preds = trainer.predict(mpnn, test_loader)

  # extract only pchembl value
  return test_preds[0][0].numpy()[0]


class pchemble_reward(Reward):
  def get_objective_functions(conf):
    # we need to hook our chemprop model here to predict a pChEMBLE value
    def get_pchemble_pred(mol):
            return predict_chemprop(mol)

    return [get_pchemble_pred]

  def calc_reward_from_objective_values(values, conf):
    pchemble_val = values[0]
    if pchemble_val < 7:
      return -1
    scaled_val = (values[0] / 10) # lets arbitrarily set max score to 10
    return scaled_val

Overwriting /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_reward.py


### Let's also create a config .yaml file

In [ ]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_setting.yaml

# Basic setting
c_val: 1.0
# threshold_type: [time, generation_num]
threshold_type: generation_num
#hours: 0.01
generation_num: 500
output_dir: result/example01
model_setting:
  model_json: model/model.tf25.json
  model_weight: model/model.tf25.best.ckpt.h5
token: model/tokens.pkl
reward_setting:
  reward_module: reward.pchemble_reward
  reward_class: pchemble_reward

# Advanced setting
expansion_threshold: 0.995
simulation_num: 3
flush_threshold: -1
policy_setting:
  policy_module: policy.ucb1
  policy_class: Ucb1

# Restart setting
save_checkpoint: False
restart: False
checkpoint_file: chemtsv2.ckpt.pkl

# Filter setting
use_lipinski_filter: True
lipinski_filter:
  module: filter.lipinski_filter
  class: LipinskiFilter
  type: rule_of_5
use_radical_filter: True
radical_filter:
  module: filter.radical_filter
  class: RadicalFilter
use_pubchem_filter: True
pubchem_filter:
  module: filter.pubchem_filter
  class: PubchemFilter
use_sascore_filter: True
sascore_filter:
  module: filter.sascore_filter
  class: SascoreFilter
  threshold: 3.5
use_ring_size_filter: True
ring_size_filter:
  module: filter.ring_size_filter
  class: RingSizeFilter
  threshold: 6
use_pains_filter: False
pains_filter:
  module: filter.pains_filter
  class: PainsFilter
  type: [pains_a]
use_covalent_warhead_filter: False
covalent_warhead_filter:
  module: filter.covalent_warhead_filter
  class: CovalentWarheadFilter
include_filter_result_in_reward: False

Overwriting /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_setting.yaml


## We've know succesfully created our reward and config file for ChemTSV. Let's now start generating structures!!


In [ ]:
!git clone https://github.com/molecule-generator-collection/ChemTSv2.git

Cloning into 'ChemTSv2'...
remote: Enumerating objects: 3196, done.
remote: Counting objects: 100% (663/663), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 3196 (delta 564), reused 524 (delta 524), pack-reused 2533 (from 2)
Receiving objects: 100% (3196/3196), 116.76 MiB | 9.86 MiB/s, done.
Resolving deltas: 100% (2074/2074), done.
Updating files: 100% (154/154), done.


In [ ]:
# copy reward and config file into the chemtsv repo
!cp /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_reward.py /content/ChemTSv2/reward/pchemble_reward.py
!cp /content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/pchemble_setting.yaml /content/ChemTSv2/config/pchemble_setting.yaml

In [ ]:
%cd ChemTSv2
!chemtsv2 -c config/pchemble_setting.yaml --gpu 0

Streaming output truncated to the last 5000 lines.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 0:00:00 • 0:00:00 0.00it/s 
Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 1, batch_size = 64)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 0:00:00 • 0:00:00 0.00it/s 
Dropping last batch of size 1 to avoid issues with batch normalization     (dat

In [ ]:
assert False

AssertionError: 

In [ ]:
# Run this cell to downlaod the best model file
from google.colab import files

results_file = !find result/example01 -name '*.csv'
results_file

# download csv file
files.download(results_file[0])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Wow! ChemTSV2 really generated 500 novel compounds! But how can we determine if these compounds are non-opioids?

For our next step let's build another chemprop model to predict opioid interactivity!!